# 03 · Embed — 05 A local vector store this stage can actually write to

**Ported from `clinical-search`'s `local_stack/chroma_store.py`, branch `pr-1` (open, unmerged) at commit `130ff868`. `02-chunk/03-store-backends.ipynb` already writes to Chroma from the chunk side; this closes the same loop on the embed side, so a contributor with no key and no network can embed, store, and search a real (if hash-embedded) index end to end.**

`search_chroma()` and `upsert_chunks()`, adapted in one way: the donor calls
a product-specific local-embedding module (Ollama `nomic-embed-text`,
requiring a model download); this notebook takes an `embed_fn` parameter
instead, defaulting to notebook 04's offline `_hash_embed` — no download,
consistent with this repo's "run in 60 seconds, no key" promise. The
storage and search logic itself is unchanged.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `get_collection` | Resolves (or creates) a per-namespace Chroma collection, cached per kernel | `get_collection("demo")` |
| `upsert_chunks` | Embeds and stores chunk texts + metadata | `upsert_chunks("demo", ids, texts, metas, embed_fn=hash_embed_batch)` |
| `search_chroma` | Embeds a query, returns scored, deduplicated results with a resolved source link | `search_chroma("demo", "mitochondria", embed_fn=hash_embed)` |


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        sys.path.insert(0, str(_root))
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — sqlite3 patch (same gotcha as `02-chunk/03-store-backends.ipynb`)

`chromadb` requires sqlite3 >= 3.35.0; some Linux clusters ship an older
system sqlite3. Guarded so it's a no-op where the system sqlite3 is
already new enough — see that notebook for the full explanation, not
repeated here.

In [ ]:
import sys

try:
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

import chromadb
print("chromadb", chromadb.__version__, "- sqlite3", __import__("sqlite3").sqlite_version)

## Step 2 — the offline embedding function this notebook uses

Same `_hash_embed` shape as notebook 04 / `03-embed/01-offline-embeddings.ipynb`
-- real wiring, not semantically meaningful vectors. Passed in as
`embed_fn`, the one adaptation from the donor: production calls Ollama
directly inside `chroma_store.py`; this notebook keeps the store logic
provider-agnostic and injects the provider instead, so it works with
notebook 04's `embed_texts()` too, not only the hash stub.

In [ ]:
import hashlib
import math


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def hash_embed_batch(texts: list[str], dim: int = 384) -> list[list[float]]:
    return [hash_embed(t, dim) for t in texts]


print(f"hash_embed dim={len(hash_embed('test'))}")

## Step 3 — `get_collection`: one per-namespace collection, cached per kernel

Ported from `_get_collection`, generalized: the donor resolves a
"department id" via a product-specific domain loader; this notebook takes
a plain namespace string instead, since the cookbook has no department
concept. Written to a temp directory, not a path inside the repo -- this
is a demo index, not one meant to persist or be committed.

In [ ]:
import tempfile

_CHROMA_DIR = Path(tempfile.mkdtemp(prefix="cookbook-chroma-"))
_collections: dict[str, object] = {}


def get_collection(namespace: str):
    if namespace in _collections:
        return _collections[namespace]
    path = str(_CHROMA_DIR / namespace)
    client = chromadb.PersistentClient(path=path)
    collection = client.get_or_create_collection(name=f"cookbook_{namespace}")
    _collections[namespace] = collection
    return collection


demo_collection = get_collection("demo")
print(f"collection {demo_collection.name!r}, {demo_collection.count()} vectors so far")

## Step 4 — `doc_link`: resolve one source link from whatever citation fields exist

Ported unchanged. A chunk's metadata might carry a PMC URL, a DOI URL, or
a bare DOI — this picks whichever is present and normalizes a bare DOI
into a real clickable link, so a caller never has to branch on which
citation field happened to be populated.

In [ ]:
def doc_link(meta: dict) -> str:
    pmc_url = str(meta.get("pmc_url") or "").strip()
    if pmc_url:
        return pmc_url
    doi_url = str(meta.get("doi_url") or "").strip()
    if doi_url:
        return doi_url
    doi = str(meta.get("doi") or "").strip()
    if doi.startswith("http"):
        return doi
    if doi.startswith("10."):
        return f"https://doi.org/{doi}"
    return ""


print(doc_link({"doi": "10.1000/burns.123"}))
print(doc_link({"pmc_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC123/"}))
print(doc_link({}))

## Step 5 — `upsert_chunks`: embed and store

Ported from `upsert_chunks`, with `embed_fn` injected rather than a
hardcoded Ollama call (see Step 2). Two synthetic chunks, upserted into
the `demo` namespace.

In [ ]:
def upsert_chunks(namespace: str, chunk_ids: list[str], texts: list[str], metadatas: list[dict], embed_fn=hash_embed_batch) -> int:
    collection = get_collection(namespace)
    vectors = embed_fn(texts)
    collection.upsert(ids=chunk_ids, embeddings=vectors, documents=texts, metadatas=metadatas)
    return len(chunk_ids)


chunk_ids = ["c0", "c1"]
texts = [
    "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
]
metadatas = [
    {"title": "Cell biology basics", "doi": "10.1000/cellbio.1", "year": "2020"},
    {"title": "Plant biology basics", "doi": "10.1000/plantbio.1", "year": "2019"},
]

n = upsert_chunks("demo", chunk_ids, texts, metadatas)
print(f"upserted {n} chunks; collection now holds {get_collection('demo').count()} vectors")

## Step 6 — `search_chroma`: embed a query, return scored real results

Ported from `search_chroma`. Distance -> score conversion
(`score = 1.0 - distance`), year/citation-count coercion guarded against
malformed metadata, and `doc_link` applied per result — all unchanged from
the donor.

In [ ]:
def search_chroma(namespace: str, query: str, top_k: int = 5, embed_fn=hash_embed) -> list[dict]:
    collection = get_collection(namespace)
    query_vector = embed_fn(query)
    results = collection.query(query_embeddings=[query_vector], n_results=top_k)

    docs = []
    ids = results.get("ids", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    documents = results.get("documents", [[]])[0]

    for i, chunk_id in enumerate(ids):
        meta = metadatas[i] if i < len(metadatas) else {}
        text = documents[i] if i < len(documents) else ""
        distance = distances[i] if i < len(distances) else None
        score = (1.0 - distance) if distance is not None else None
        docs.append({
            "text": text,
            "chunk_id": chunk_id,
            "score": score,
            "title": meta.get("title", ""),
            "doi": meta.get("doi", ""),
            "source_link": doc_link(meta),
            "source": "chromadb",
        })
    return docs


# A hash embedding has no notion of meaning (same honest caveat as
# 01-offline-embeddings.ipynb) -- it CANNOT be trusted to rank "what
# produces ATP in a cell?" above the mitochondria chunk over the
# photosynthesis one, and asserting that would assert something this
# stub doesn't provide. What IS guaranteed: querying with a chunk's own
# exact stored text must rank that chunk first, since cosine similarity
# to itself is the maximum possible score. That's what's checked below.
results = search_chroma("demo", "what produces ATP in a cell?", top_k=2)
for r in results:
    print(f"  score={r['score']:.3f}  {r['title']!r}  {r['text'][:55]!r}")
print("(ranking above is NOT semantically meaningful -- see the note above)")
print()

self_query_results = search_chroma("demo", texts[0], top_k=2, embed_fn=hash_embed)
print("querying with chunk 0's own exact text:")
for r in self_query_results:
    print(f"  score={r['score']:.3f}  {r['chunk_id']}")

assert self_query_results[0]["chunk_id"] == "c0", "a chunk's own text must retrieve itself first"
assert abs(self_query_results[0]["score"] - 1.0) < 1e-6, "self-similarity must be the maximum possible score"
print()
print("confirmed: self-retrieval works and scores exactly 1.0 -- the store and search wiring is correct")


## Where this fits

`02-chunk/03-store-backends.ipynb` writes chunks into Chroma; this
notebook is the read side, on the embed stage, using the same store —
the two together are the full offline write-then-search loop with no key,
no network, and no server.

## What did not come across

`_get_department_id()`'s domain-loader lookup — replaced with a plain
`namespace` string parameter, since the cookbook has no department
concept. `local_embeddings.embed_query_local` / `embed_batch_local`
(Ollama `nomic-embed-text`) — not copied; `embed_fn` is injectable
instead, and the offline default needs no model download. Wiring in a
real local model, if wanted, is a matter of passing a different
`embed_fn` — the store and search logic above does not change.